# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moriyamao/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

*Skill loaded: `writing-honest-claims` + `flyrank/flyrank-data`.*

This playbook blends the Week-4 rule baseline with the Week-6 honestly-validated model (client-grouped split) into one ranked action queue. It only scores the **held-out slice** the model was never trained on — no page in this playbook was in the model's training set, so nothing here is in-sample.

In [1]:
import os
if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
    !git clone https://github.com/moriyamao/flyrank-ml-internship.git repo
    os.chdir('repo')

import pandas as pd
import numpy as np
import json
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

num_features = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'engagement_rate', 'scroll_rate',
                'ai_traffic_pct', 'avg_position', 'ctr', 'word_count', 'content_age_days',
                'days_since_last_update', 'search_volume', 'competition']
cat_features = ['content_type', 'main_intent', 'competition_level']
X = df[num_features + cat_features]
y = df['is_declining_label']
groups = df['client_id']

# Same client-grouped split as Weeks 5-6 (random_state=42, reproducible).
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

pre = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), num_features),
    ('cat', Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='unknown')),
                       ('oh', OneHotEncoder(handle_unknown='ignore'))]), cat_features)
])
rf = Pipeline([('pre', pre), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1))])
rf.fit(X.iloc[train_idx], y.iloc[train_idx])

df['model_proba'] = np.nan
df.loc[df.index[test_idx], 'model_proba'] = rf.predict_proba(X.iloc[test_idx])[:, 1]
print('Honestly-scored (held-out) rows:', df['model_proba'].notna().sum(), 'of', len(df))

Cloning into 'repo'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (127/127), done.
remote: Total 172 (delta 73), reused 95 (delta 29), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 1.88 MiB | 10.29 MiB/s, done.
Resolving deltas: 100% (73/73), done.
Honestly-scored (held-out) rows: 7115 of 30000


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**The blend:** 50% Week-4 rule score (normalized 0–1) + 50% Week-6 honest model probability. The rule contributes transparency (a human can always see *why*); the model contributes the small, directional lift over the rule that Week 6 measured, without letting either one dominate.

**Reason codes** (pipe-separated, a page can carry more than one): `stale_but_visible`, `ctr_gap_for_position`, `model_decline_risk` (model probability ≥ 0.65).

**Actions:** `refresh_and_fix_ctr`, `refresh`, `fix_ctr`, `monitor_closely` (model flags it, the rule doesn't — worth a second look, not an automatic queue entry), `monitor`.

**Archetype → action mapping** (named groups from the flag combination, in the spirit of the paper's content archetypes, but built from this lane's own flags rather than borrowed cluster names):

| Archetype | Meaning | Typical action |
|---|---|---|
| Stale + CTR Gap | Both signals agree | `refresh_and_fix_ctr` — the highest-confidence group |
| Stale Only | Content aging, CTR is fine for its position | `refresh` |
| CTR Gap Only | Ranking is fine, clicks are underperforming for that position | `fix_ctr` |
| Model-Flagged (rule silent) | Model risk ≥0.65 but neither rule flag tripped | `monitor_closely` — flag for a human, don't auto-queue |
| No Flag | Neither signal present | `monitor` |

**The decay/refresh insight, stated carefully:** in this dataset, pages 91+ days since their last update show a higher observed decline rate (61.1%, n=9,171) than pages updated in the last 30 days (51.1%, n=20,480) — see Week 4's signal check. This is an **observed, directional** pattern in one snapshot, not a guarantee that any individual refresh will reverse decline.

In [2]:
visible = df['impressions_90d'] >= 300
stale = df['freshness_tier'].isin(['91-180', '181+'])
tier_median_ctr = df.groupby('position_tier')['ctr'].transform('median')
ctr_gap = (df['ctr'] < tier_median_ctr) & (df['position_tier'].isin(['top_3', 'page_1', 'striking']))
df['stale_flag'] = stale.astype(int)
df['ctr_gap_flag'] = ctr_gap.astype(int)
df['rule_score'] = visible.astype(int) * (df['stale_flag'] + df['ctr_gap_flag']) * np.log1p(df['impressions_90d'])

# Playbook universe: ONLY the held-out slice the model was never trained on -- no in-sample scoring.
scored = df[df['model_proba'].notna()].copy()

def archetype(row):
    if row['stale_flag'] and row['ctr_gap_flag']:
        return 'Stale + CTR Gap'
    if row['stale_flag']:
        return 'Stale Only'
    if row['ctr_gap_flag']:
        return 'CTR Gap Only'
    if row['model_proba'] >= 0.65:
        return 'Model-Flagged (rule silent)'
    return 'No Flag'

def reason_code(row):
    parts = []
    if row['stale_flag']: parts.append('stale_but_visible')
    if row['ctr_gap_flag']: parts.append('ctr_gap_for_position')
    if row['model_proba'] >= 0.65: parts.append('model_decline_risk')
    return '|'.join(parts) if parts else 'no_flag'

def action_label(row):
    if row['stale_flag'] and row['ctr_gap_flag']:
        return 'refresh_and_fix_ctr'
    if row['stale_flag']:
        return 'refresh'
    if row['ctr_gap_flag']:
        return 'fix_ctr'
    if row['model_proba'] >= 0.65:
        return 'monitor_closely'
    return 'monitor'

scored['archetype'] = scored.apply(archetype, axis=1)
scored['reason_code'] = scored.apply(reason_code, axis=1)
scored['action'] = scored.apply(action_label, axis=1)

rs = scored['rule_score']
rule_norm = (rs - rs.min()) / (rs.max() - rs.min() + 1e-9)
scored['blended_score'] = 0.5 * rule_norm + 0.5 * scored['model_proba'].fillna(0)

queue = scored.sort_values('blended_score', ascending=False).reset_index(drop=True)
print('Playbook universe (held-out, honest):', len(queue))
print()
print(queue['archetype'].value_counts())
print()
print(queue['action'].value_counts())

Playbook universe (held-out, honest): 7115

archetype
CTR Gap Only                   2546
No Flag                        2449
Model-Flagged (rule silent)     855
Stale Only                      774
Stale + CTR Gap                 491
Name: count, dtype: int64

action
fix_ctr                2546
monitor                2449
monitor_closely         855
refresh                 774
refresh_and_fix_ctr     491
Name: count, dtype: int64


In [3]:
top10 = queue.head(10)[['content_id', 'client_id', 'blended_score', 'rule_score', 'model_proba',
                         'archetype', 'reason_code', 'action']]
top10

,content_id,client_id,blended_score,rule_score,model_proba,archetype,reason_code,action
0,content_5fe46e04994d,client_4e07408562,0.764892,26.314364,0.529784,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position,refresh_and_fix_ctr
1,content_ec521b760d7c,client_8527a891e2,0.745382,19.269647,0.758477,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position|model_d...,refresh_and_fix_ctr
2,content_a5dbb404bdc2,client_f369cb89fc,0.743227,22.555317,0.629306,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position,refresh_and_fix_ctr
3,content_c65ee459f729,client_f369cb89fc,0.726387,17.567405,0.785176,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position|model_d...,refresh_and_fix_ctr
4,content_5ce1a9d3e4d7,client_8527a891e2,0.714145,17.993552,0.744499,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position|model_d...,refresh_and_fix_ctr
5,content_6ac3ab740bbf,client_f369cb89fc,0.709608,20.039250,0.657684,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position|model_d...,refresh_and_fix_ctr
6,content_41baf0722ad9,client_8527a891e2,0.709072,16.088611,0.806744,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position|model_d...,refresh_and_fix_ctr
7,content_98aa0aecb1d9,client_8527a891e2,0.708603,17.109363,0.767016,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position|model_d...,refresh_and_fix_ctr
8,content_df1e8cee858c,client_f369cb89fc,0.704261,18.251525,0.714927,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position|model_d...,refresh_and_fix_ctr
9,content_4d9f36001f06,client_8527a891e2,0.702083,16.245336,0.786809,Stale + CTR Gap,stale_but_visible|ctr_gap_for_position|model_d...,refresh_and_fix_ctr


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Intended use:** a content strategist or SEO manager uses this ranked list as a **starting point for a weekly review queue** — which pages to look at first when deciding refresh/CTR-fix priorities, not a list of pages to act on automatically. It is decision-support, not decision-making.

**Where it stops being valid:**
- **Snapshot, not longitudinal.** The starter CSV is a single trailing-90-day export. The playbook says nothing about a page's trajectory before or after this window beyond the 30d-vs-prev-30d trend already baked into the label. It does not track a page over multiple refresh cycles.
- **One 30k-row teaching slice, 32 clients.** Findings (including the staleness and CTR-position patterns) are observed in this sample; they are not validated against the full 79M-row warehouse and should not be assumed to hold at that scale without re-checking.
- **Model score is honest but modest.** Week 6 confirmed the client-grouped AUC is 0.600 — real signal, not strong signal. The blended score should be read as "worth a second look," never as a confidence percentage.
- **Not causal.** Nothing here says refreshing a page *will* reverse its decline — the staleness/decline association is observed and directional, from cross-sectional data with no experiment behind it.
- **Only the held-out 7,115 rows are scored honestly.** The other ~23k training rows are deliberately excluded from this playbook — scoring them would be in-sample and would inflate confidence artificially.

In [4]:
print('Playbook covers only the honest held-out slice:', len(queue), 'of', len(df), 'total rows')
print('Excluded (used for model training, would be in-sample if scored):', len(df) - len(queue))

Playbook covers only the honest held-out slice: 7115 of 30000 total rows
Excluded (used for model training, would be in-sample if scored): 22885


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on any queue entry, a human must check:**
1. Does the page's low CTR trace to a real content problem, or a SERP feature (featured snippet, PAA box) stealing the click? The rule and model can't tell these apart — Week 4's weakest pick (a 0.00% CTR entry) is the canonical example.
2. Is the page's staleness a genuine content-aging signal, or a CMS/migration timestamp artifact?
3. Does the target keyword's intent still match the page's content, or has search intent for that query shifted since the page was written?
4. Is there a legitimate business reason the page underperforms (seasonal product, deprecated offering) that no refresh should chase?

**What should NEVER be automated:**
- Auto-publishing or auto-editing content based on this queue — every action is a **recommendation**, not a content-management-system trigger.
- Deleting or de-indexing pages from this queue alone — that decision needs a human who has checked traffic history and business context beyond this 90-day snapshot.
- Treating `model_decline_risk` as a standalone trigger — the `Model-Flagged (rule silent)` archetype exists specifically to route those 855 pages to human review, not to an automatic action, since the model has no transparent reason code the way the rule does.
- Any action for a page below the visibility floor (`impressions_90d` < 300) — too little signal to trust either the rule or the model at that volume.
- Client-facing reporting of the blended score as a percentage/probability of success — it is a ranking signal, not a calibrated probability of a business outcome.

In [5]:
# No-go floor, made concrete: how many held-out rows fall below the visibility floor
# and therefore should never receive an automated action regardless of score.
below_floor = (scored['impressions_90d'] < 300).sum()
print('Rows below the 300-impression visibility floor (no-go for automation):', below_floor, 'of', len(scored))

Rows below the 300-impression visibility floor (no-go for automation): 3575 of 7115


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Light monitoring plan** (practical for a non-production teaching project, not a real deployed system):

- **Base-rate drift.** Re-check `is_declining_label` mean on a fresh export. Week 6's numbers assume a base rate near 0.52; if a new export's base rate moves more than ~10 points, the model's calibration should be treated as stale until re-validated.
- **Precision@50 spot-check.** If a fresh, labeled sample of 50+ pages shows the top-ranked queue entries performing meaningfully below the 0.56 grouped-split precision@50 measured in Week 6, that's a retrain signal, not a one-off bad batch — check at least two review cycles before concluding drift.
- **Reason-code health.** If `Model-Flagged (rule silent)` starts dominating the queue (currently 855 of 7,115, ~12%), that means the model and the transparent rule are diverging — worth investigating before trusting the model half of the blend further.
- **Retrain trigger:** any of the above, OR a full quarter passing (content, SERP features, and client mix all drift over a quarter even in a static teaching dataset's real-world analogue) — whichever comes first.
- **What this monitoring plan is NOT:** an automated pipeline. For this project, monitoring means a human re-runs this notebook's Section 1–3 numbers periodically and compares them by eye — there is no scheduled job.

In [6]:
# Snapshot of the numbers a future monitoring check should compare against.
monitoring_baseline = {
    'base_rate': round(float(y.iloc[test_idx].mean()), 3),
    'grouped_split_auc': 0.600,
    'grouped_split_precision_at_50': 0.560,
    'model_flagged_rule_silent_share': round(len(queue[queue['archetype'] == 'Model-Flagged (rule silent)']) / len(queue), 3),
}
monitoring_baseline

{'base_rate': 0.517,
 'grouped_split_auc': 0.6,
 'grouped_split_precision_at_50': 0.56,
 'model_flagged_rule_silent_share': 0.12}

## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [7]:
os.makedirs('work/outputs', exist_ok=True)
os.makedirs('work/figures', exist_ok=True)

export_cols = ['content_id', 'client_id', 'blended_score', 'rule_score', 'model_proba',
               'archetype', 'reason_code', 'action', 'impressions_90d', 'ctr', 'avg_position',
               'freshness_tier', 'position_tier']
queue[export_cols].to_csv('work/outputs/content_action_playbook.csv', index=False)
print('Written: work/outputs/content_action_playbook.csv (', len(queue), 'rows )')

# Metrics JSON -- the receipts the paper's numbers trace back to. This one IS committed.
metrics = {
    'playbook_universe_n': int(len(queue)),
    'total_rows_n': int(len(df)),
    'archetype_counts': queue['archetype'].value_counts().to_dict(),
    'action_counts': queue['action'].value_counts().to_dict(),
    'grouped_split_auc': 0.600,
    'grouped_split_precision_at_50': 0.560,
    'random_split_auc_for_comparison': 0.745,
    'below_visibility_floor_n': int(below_floor),
    'random_seed': 42,
}
with open('work/outputs/playbook_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Written: work/outputs/playbook_metrics.json')
print(json.dumps(metrics, indent=2))

Written: work/outputs/content_action_playbook.csv ( 7115 rows )
Written: work/outputs/playbook_metrics.json
{
  "playbook_universe_n": 7115,
  "total_rows_n": 30000,
  "archetype_counts": {
    "CTR Gap Only": 2546,
    "No Flag": 2449,
    "Model-Flagged (rule silent)": 855,
    "Stale Only": 774,
    "Stale + CTR Gap": 491
  },
  "action_counts": {
    "fix_ctr": 2546,
    "monitor": 2449,
    "monitor_closely": 855,
    "refresh": 774,
    "refresh_and_fix_ctr": 491
  },
  "grouped_split_auc": 0.6,
  "grouped_split_precision_at_50": 0.56,
  "random_split_auc_for_comparison": 0.745,
  "below_visibility_floor_n": 3575,
  "random_seed": 42
}


In [8]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4))
queue['archetype'].value_counts().plot(kind='barh', ax=ax, color='#4C72B0')
ax.set_xlabel('Pages (held-out slice)')
ax.set_title('Action Playbook: pages by archetype')
plt.tight_layout()
plt.savefig('work/figures/archetype_counts.png', dpi=150)
plt.close()
print('Written: work/figures/archetype_counts.png')

Written: work/figures/archetype_counts.png


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all) — **run it yourself once to confirm**
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — commit `work/figures/archetype_counts.png` and `work/outputs/playbook_metrics.json` too (the CSV itself stays out of git by design). Then submit your repo URL on the card. Done.